In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import math
import string
import random
import time
from datetime import datetime
import unicodedata #as some text is french this is important
from collections import Counter
from pickle import load, dump
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

False

In [2]:
# #We will not be using the tokens generated from Autotokenizer
# from transformers import AutoTokenizer
# fra_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-fr-en")
# eng_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

# print(f"Special Tokens: {eng_tokenizer.all_special_tokens}")
# print(f"Special IDs: {eng_tokenizer.all_special_ids}")
# print(f"Special Tokens: {fra_tokenizer.all_special_tokens}")
# print(f"Special IDs: {fra_tokenizer.all_special_ids}")

# tokens_file = "data/tokens.pkl" # for jupyter
# with open(tokens_file, 'rb') as file:
#     tokens = load(file)

In [3]:
PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"

MIN_FREQ = 5

TABLE = str.maketrans("","",string.punctuation.replace("'","")) # as french words include '
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [4]:
seed = 42

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

np.random.seed(seed)
random.seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [11]:
# filepath = "data/eng-fra.txt"
# vocabpath = "data/vocabs.pkl"
filepath = "eng-fra.txt"
vocabpath = "vocabs.pkl"
# !ls

In [6]:
def preprocess(sent):
    sent = unicodedata.normalize("NFKC", sent)
    return sent.lower().translate(TABLE)

In [7]:
class Vocab():
    def __init__(self, lang, min_freq):
        self.stoi = {PAD:0,SOS:1,EOS:2,UNK:3}
        self.itos = {0:PAD,1:SOS,2:EOS,3:UNK}
        self.freqs = {}
        self.nxt_idx = 4
        self.lang = lang
        self.min_freq = min_freq

    def build_vocab(self, corpus): #we should get the corpus as an iterable of all the words
        freqs = Counter(corpus)
        # VERY IMP: Counter can store in random order for diff runs so sort before vocab.
        self.freqs = dict(sorted(freqs.items(), key=lambda x:x[1], reverse=True)) #higher freq -> lower index
        for word, freq in self.freqs.items():
            if freq >= self.min_freq and word not in self.stoi:
                self.stoi[word] = self.nxt_idx
                self.itos[self.nxt_idx] = word
                self.nxt_idx += 1

    def encode(self, sent):
        tokens = [self.stoi[SOS]]
        tokens += [self.stoi.get(word, self.stoi[UNK]) for word in sent]
        tokens += [self.stoi[EOS]]

        return tokens

    def decode(self, tokens):
        return " ".join([self.itos.get(token,UNK) for token in tokens[1:-1]])


In [8]:
class Fra2EngDataset(Dataset):
    def __init__(self, filepath, min_freq, vocab_path=None):
        super().__init__()
        self.fra_vocab = Vocab(lang="fra", min_freq=min_freq)
        self.eng_vocab = Vocab(lang="eng", min_freq=min_freq)
        self.pairs = []
        self.max_len = 0

        with open(filepath, 'r', encoding="utf-8") as file:
            for line in file:
                pair = line.strip().split("\t")
                if len(pair) != 2:
                    continue
                eng_tokens = preprocess(pair[0]).split()
                fra_tokens = preprocess(pair[1]).split()
                self.pairs.append((eng_tokens, fra_tokens))
                self.max_len = max([self.max_len, len(eng_tokens), len(fra_tokens)])

        if vocab_path is None:
            corpus = {"eng": [], "fra": []}
            for pair in self.pairs:
                corpus["eng"].extend(pair[0])
                corpus["fra"].extend(pair[1])

            self.eng_vocab.build_vocab(corpus["eng"])
            self.fra_vocab.build_vocab(corpus["fra"])

        else:
            with open(vocab_path, 'rb') as f:
                vocabs = load(f)
                self.eng_vocab = vocabs["eng"]
                self.fra_vocab = vocabs["fra"]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        eng_tokens = self.eng_vocab.encode(pair[0])
        fra_tokens = self.fra_vocab.encode(pair[1])

        eng_tensor = torch.tensor(eng_tokens, dtype=torch.long) #shifted to right by 1 due to SOS
        fra_tensor = torch.tensor(fra_tokens[1:], dtype=torch.long) #input to encoder so no SOS

        return fra_tensor, eng_tensor # since fra -> eng

In [9]:
# dataset = Fra2EngDataset(filepath, MIN_FREQ)
# dataset.eng_vocab.nxt_idx, dataset.fra_vocab.nxt_idx

In [12]:
# with open(vocabpath, 'wb') as f:
#     dump({"eng":dataset.eng_vocab, "fra":dataset.fra_vocab}, f)
dataset = Fra2EngDataset(filepath, MIN_FREQ, vocabpath)
dataset.eng_vocab.nxt_idx, dataset.fra_vocab.nxt_idx

(5515, 8575)

In [13]:
dataset[61110]

(tensor([431,  35,  10,  55,  46,  37, 121,   2]),
 tensor([ 1, 94, 39,  7, 51,  5, 81,  2]))

In [14]:
def custom_padding(batch, pad_idx = dataset.eng_vocab.stoi[PAD]):
    fra, eng = zip(*batch)

    max_fra = max(x.size(0) for x in fra)
    max_eng = max(x.size(0) for x in eng)

    fra_batch = torch.full((len(batch), max_fra), pad_idx, dtype=torch.long)
    eng_batch = torch.full((len(batch), max_eng), pad_idx, dtype=torch.long)

    for i in range(len(batch)):
        fra_batch[i, :fra[i].size(0)] = fra[i]
        eng_batch[i, :eng[i].size(0)] = eng[i]

    return fra_batch, eng_batch

In [15]:
train_loader = DataLoader(dataset, collate_fn=custom_padding, drop_last=True, batch_size=32, num_workers=2)

In [16]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divible by num heads."

        self.d_model = d_model
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.W_v = nn.Linear(self.d_model, self.d_model, bias=False)
        self.W_k = nn.Linear(self.d_model, self.d_model, bias=False)
        self.W_o = nn.Linear(self.d_model, self.d_model, bias=False)

    def split_heads(self, x):
        # x = [bs, sl, d] -> [bs, sl, nh, dk] -> [bs, nh, sl, dk]
        batch_size, seq_len, _ = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)

    def combine_heads(self, x):
        # x = [bs, nh, sl, dk] -> [bs, sl, nh, dk] -> [bs, sl, d]
        batch_size, _, seq_len, _ = x.size()
        return x.transpose(1,2).contiguous().view(batch_size, seq_len, self.d_model)

    def scaled_dot_product_attn(self, Q, K, V, mask=None):
        # Q = [bs, nh, sl, dk]
        alignment_scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # alignment_scores = [bs, nh, sl, sl] = [bs, nh, querylen, keylen]

        if mask is not None: # apply mask
            alignment_scores = alignment_scores.masked_fill(mask == 0, float('-1e9'))

        attn_weights = torch.softmax(alignment_scores, dim=-1)
        contextual_embed = attn_weights @ V
        # contextual_embed = [bs, nh, sl, dk]

        return contextual_embed, attn_weights

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        embed, attn_wts = self.scaled_dot_product_attn(Q, K, V, mask)
        output = self.W_o(self.combine_heads(embed))

        return output

In [17]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.layer1 = nn.Linear(d_model, d_ff)
        self.layer2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

In [18]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len):
        super().__init__()

        # PE = sin or cos(pos / 10000^(2i/d))
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        self.pe = torch.zeros(max_seq_len, d_model)

        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000) / d_model)) #this is the denominator in angle
        self.pe[:, 0::2] = torch.sin(position * div_term) # position = [msl, 1], div = [d/2]
        self.pe[:, 1::2] = torch.cos(position * div_term)

    def forward(self, x):
        # x = embedding = [bs, sl, d]
        return x + self.pe[:x.size(1), :]

In [19]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, ):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForwardNetwork(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=0.1)

    def forward(self, x, mask=None):
        # input x = embed + positional encoding = [bs, sl, d]
        z = self.mha(x, x, x, mask)
        z_norm1 = self.norm1(self.dropout(z) + x) # residual connections
        z = self.ffn(z_norm1)
        z_norm2 = self.norm2(self.dropout(z) + z_norm1)

        return z_norm2

In [20]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, ):
        super().__init__()

        self.masked_mha = MultiHeadAttention(d_model, num_heads)
        self.cross_mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForwardNetwork(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # src_mask = fra mask = padding only
        # tgt_mask = eng mask = causal + padding mask
        # non auto regressive in training

        z = self.masked_mha(x, x, x, tgt_mask)
        z1 = self.norm1(self.dropout(z) + x)
        z2 = self.cross_mha(x, enc_output, enc_output, src_mask)
        z3 = self.norm2(self.dropout(z2) + z1)
        y = self.ffn(z3)
        out = self.norm3(self.dropout(y) + z3)

        return out

In [21]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads, num_layers, d_ff, max_seq_len, src_vocab_size, tgt_vocab_size):
        super().__init__()

        # input
        self.src_embed = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)

        # blocks
        self.encoder_block = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.decoder_block = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)])

        #output
        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(0.1)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        # print(type(src_mask))

        seqlen = tgt.size(1)
        no_peak_mask = (1 - torch.triu(torch.ones(1, seqlen, seqlen), diagonal=1)).bool()

        tgt_mask = tgt_mask & no_peak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):

        src_mask, tgt_mask = self.generate_mask(src, tgt)
        input_src = self.dropout(self.positional_encoding(self.src_embed(src)))
        input_tgt = self.dropout(self.positional_encoding(self.tgt_embed(tgt)))

        enc_out = input_src
        for enc_layer in self.encoder_block:
            # print(type(enc_out))
            enc_out = enc_layer(enc_out, src_mask)

        dec_out = input_tgt
        for dec_layer in self.decoder_block:
            dec_out = dec_layer(dec_out, enc_out, src_mask, tgt_mask)

        output = self.fc(dec_out)
        return output

In [22]:
D_MODEL = 128
NUM_HEADS = 4
NUM_LAYERS = 2
D_FF = 512
SRC_VOCAB_SIZE = dataset.fra_vocab.nxt_idx
TGT_VOCAB_SIZE = dataset.eng_vocab.nxt_idx
MAX_SEQ_LEN = dataset.max_len
EPOCHS = 75

In [23]:
model = Transformer(D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF, MAX_SEQ_LEN, SRC_VOCAB_SIZE, TGT_VOCAB_SIZE)

In [24]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [25]:
src1, tgt1 = next(iter(train_loader))
# model(src1, tgt1)

In [26]:
def train_one_epoch(model, criterion, optimizer, train_loader):
    progress = tqdm(train_loader, desc="Data", total=len(train_loader))
    train_losses = []

    model.train()

    for src, tgt in progress:
        src, tgt = src.to(device), tgt.to(device)
        # src = [bs, sl], tgt = [bs, sl]

        output = model(src, tgt[:, :-1]) # skip EOS as we dont want to predict anything by sending this as input
        predicted = output.contiguous().view(-1, TGT_VOCAB_SIZE) # output = [bs, sl, tgt_vocab_size]
        target = tgt[:,1:].contiguous().view(-1) # compare w1 with w1 not SOS
        loss = criterion(predicted, target)
        train_losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    return sum(train_losses) / len(train_losses)

In [27]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [28]:
save_dir = '/content/drive/MyDrive/transformer'
os.makedirs(save_dir, exist_ok=True)

In [29]:
def save_checkpoint(model, optimizer, epoch, train_losses):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': train_losses
    }
    filepath = os.path.join(save_dir, f"checkpt{epoch}.pth")
    torch.save(checkpoint, filepath)

    old_checkpoint = os.path.join(save_dir, f"checkpt{epoch-1}.pth")
    if os.path.exists(old_checkpoint):
        os.remove(old_checkpoint)
    print(f"Checkpoint saved to {filepath} at epoch {epoch}")

In [30]:
def load_checkpoint(model, optimizer, checkpt_path):
    checkpt = torch.load(checkpt_path, map_location=device)
    model.load_state_dict(checkpt['model_state_dict'])
    optimizer.load_state_dict(checkpt['optimizer_state_dict'])
    epoch = checkpt['epoch']
    losses = checkpt['loss']

    return model, optimizer, epoch, losses

In [31]:
model, optimizer, i, training_losses = load_checkpoint(model, optimizer, os.path.join(save_dir, f"checkpt11.pth"))

In [ ]:
model.to(device)
# training_losses = []

for epoch in range(i, EPOCHS+1):
    strt = time.time()
    print(f"Start time: {datetime.now()}")
    loss = train_one_epoch(model, criterion, optimizer, train_loader)
    end = time.time()
    training_losses.append(loss)
    print(f"Epoch {epoch}: Loss = {loss} | Time Taken = {end-strt} seconds")
    save_checkpoint(model, optimizer, epoch, training_losses)

Start time: 2026-03-22 13:11:52.181998


Data: 100%|██████████| 4245/4245 [08:30<00:00,  8.32it/s]


Epoch 11: Loss = 3.327853023317874 | Time Taken = 510.3486340045929 seconds
Checkpoint saved to /content/drive/MyDrive/transformer/checkpt11.pth at epoch 11
Start time: 2026-03-22 13:20:22.599276


Data:   0%|          | 0/4245 [00:00<?, ?it/s]


Epoch 11: Loss = 3.327853023317874 | Time Taken = 510.3486340045929 seconds
Checkpoint saved to /content/drive/MyDrive/transformer/checkpt11.pth at epoch 11
Start time: 2026-03-22 13:20:22.599276


...

Data: 100%|██████████| 4245/4245 [08:33<00:00,  8.26it/s]



Epoch 16: Loss = 3.2171112143951253 | Time Taken = 513.8354260921478 seconds
Checkpoint saved to /content/drive/MyDrive/transformer/checkpt16.pth at epoch 16
Start time: 2026-03-22 14:02:51.646378
Epoch 16: Loss = 3.2171112143951253 | Time Taken = 513.8354260921478 seconds
Checkpoint saved to /content/drive/MyDrive/transformer/checkpt16.pth at epoch 16
Start time: 2026-03-22 14:02:51.646378


Data: 100%|██████████| 4245/4245 [08:29<00:00,  8.34it/s]



Epoch 17: Loss = 3.2005290306358654 | Time Taken = 509.1110188961029 seconds
Checkpoint saved to /content/drive/MyDrive/transformer/checkpt17.pth at epoch 17
Start time: 2026-03-22 14:11:20.826840
Epoch 17: Loss = 3.2005290306358654 | Time Taken = 509.1110188961029 seconds
Checkpoint saved to /content/drive/MyDrive/transformer/checkpt17.pth at epoch 17
Start time: 2026-03-22 14:11:20.826840


Data:  31%|███       | 1300/4245 [02:15<05:02,  9.73it/s]

In [ ]:
train_list = list(train_loader)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
